## 1. Runtime, Drive and paths

In [ ]:
import os
from huggingface_hub import login



HF_TOKEN = ""
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import os, sys, json, math, time, random, gc, re, pickle, hashlib, zipfile, shutil, subprocess
from pathlib import Path

try:                                                    # Colab
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    MYDRIVE = Path('/content/drive/MyDrive')
    COLAB_WORK_DIR = Path('/content/phase3_rr_work')
except ImportError:                                     # local run: set PHASE3_ROOT or run from the folder holding the files
    IN_COLAB = False
    MYDRIVE = Path(os.environ.get('PHASE3_ROOT', '.')).resolve()
    COLAB_WORK_DIR = MYDRIVE / '_phase3_work'

# ---------------- EDIT HERE if your files are named / stored differently ----------------
ENCODER_CKPT_NAME  = 'best_encoder_mlm.pt'              # saved by phase1_encoder_only.ipynb
TOKENIZER_NAME     = 'legal_bpe_a1_50k_final.json'
ENCODER_CKPT_OVERRIDE = None                            # e.g. '/content/drive/MyDrive/phase1_encoder_checkpoints/best_encoder_mlm.pt'
TOKENIZER_OVERRIDE    = None
# -----------------------------------------------------------------------------------------

def find_file(name, override=None):
    if override is not None:
        p = Path(override)
        if not p.is_file():
            raise FileNotFoundError(f'Override path does not exist: {p}')
        return p
    quick = [MYDRIVE / name, MYDRIVE / 'phase1_encoder_checkpoints' / name, MYDRIVE / 'tokenizers_a1' / name]
    for p in quick:
        if p.is_file():
            return p
    found = sorted({p for p in MYDRIVE.rglob(name) if p.is_file()}, key=lambda p: p.stat().st_mtime, reverse=True)
    if not found:
        raise FileNotFoundError(f'{name} not found under {MYDRIVE}. Upload it or set the *_OVERRIDE variable above.')
    if len(found) > 1:
        print(f'NOTE: {len(found)} candidates for {name}; using the most recently modified one:')
        for p in found:
            print('   ', p)
    return found[0]

PHASE1_CHECKPOINT_PATH = find_file(ENCODER_CKPT_NAME, ENCODER_CKPT_OVERRIDE)
PHASE1_TOKENIZER_PATH  = find_file(TOKENIZER_NAME, TOKENIZER_OVERRIDE)

PROJECT_DIR = MYDRIVE / 'phase3_rr_encoder_lora'
EXPERIMENTS_ROOT = PROJECT_DIR / 'experiments'
FINAL_ARTIFACT_DIR = PROJECT_DIR / 'final_submission'
DATA_DIR = COLAB_WORK_DIR / 'data'
for _p in [PROJECT_DIR, EXPERIMENTS_ROOT, FINAL_ARTIFACT_DIR, COLAB_WORK_DIR, DATA_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

print('Phase 1 encoder checkpoint:', PHASE1_CHECKPOINT_PATH)
print('Phase 1 tokenizer         :', PHASE1_TOKENIZER_PATH)
print('Persistent output         :', PROJECT_DIR)

Mounted at /content/drive
Phase 1 encoder checkpoint: /content/drive/MyDrive/best_encoder_mlm.pt
Phase 1 tokenizer         : /content/drive/MyDrive/legal_bpe_a1_50k_final.json
Persistent output         : /content/drive/MyDrive/phase3_rr_encoder_lora


## 2. Data — download, normalise, audit (same as the GQA v2 notebook)

In [4]:
from huggingface_hub import hf_hub_download
from collections import Counter, defaultdict

hf_token = os.environ.get('HF_TOKEN')                   # optional; the public dataset needs none
if IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or hf_token
    except Exception:
        pass

repo_id = 'Exploration-Lab/CS-781-Capstone'
remote_files = {
    'cl_train': 'Phase_3_RR/Train/CL_train.json', 'cl_dev': 'Phase_3_RR/Train/CL_dev.json',
    'it_train': 'Phase_3_RR/Train/IT_train.json', 'it_dev': 'Phase_3_RR/Train/IT_dev.json',
    'label_vocab': 'Phase_3_RR/Train/label_vocab.json',
    'cl_test': 'Phase_3_RR/Test/CL_test.json', 'it_test': 'Phase_3_RR/Test/IT_test.json',
}
downloaded_paths = {}
for name, filename in remote_files.items():
    path = hf_hub_download(repo_id=repo_id, repo_type='dataset', filename=filename,
                           local_dir=str(DATA_DIR), token=hf_token or None)
    downloaded_paths[name] = Path(path)
    print(f'{name:12s} | {downloaded_paths[name].stat().st_size:,} bytes')

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

cl_train     | 2,460,243 bytes


CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

cl_dev       | 336,269 bytes


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

it_train     | 1,509,371 bytes


... (4 outputs left out)

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

cl_test      | 280,607 bytes


IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

it_test      | 194,391 bytes


In [5]:
def load_json_file(path):
    text = Path(path).read_text(encoding='utf-8')
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return [json.loads(line) for line in text.splitlines() if line.strip()]

def normalize_documents(raw, source_name):
    if isinstance(raw, list):
        records = raw
    elif isinstance(raw, dict):
        if 'text' in raw or 'sentences' in raw:
            records = [raw]
        elif isinstance(raw.get('documents'), list):
            records = raw['documents']
        elif isinstance(raw.get('data'), list):
            records = raw['data']
        else:
            records = []
            for doc_id, value in raw.items():
                if isinstance(value, dict) and ('text' in value or 'sentences' in value):
                    rec = dict(value); rec.setdefault('id', str(doc_id)); records.append(rec)
                elif isinstance(value, list) and all(isinstance(s, str) for s in value):
                    records.append({'id': str(doc_id), 'text': value})
                else:
                    raise ValueError(f'Unrecognised document format in {source_name}.')
    else:
        raise TypeError(f'Unexpected top-level type in {source_name}: {type(raw).__name__}')
    docs = []
    for index, record in enumerate(records):
        doc = dict(record)
        if 'text' not in doc and 'sentences' in doc:
            doc['text'] = doc['sentences']
        if 'id' not in doc:
            doc['id'] = doc.get('document_id', doc.get('doc_id'))
            if doc['id'] is None:
                raise ValueError(f'Document {index} in {source_name} has no id.')
        assert isinstance(doc.get('text'), list) and all(isinstance(s, str) for s in doc['text']), f'bad text in {doc["id"]}'
        doc['id'] = str(doc['id'])
        docs.append(doc)
    return docs

raw_data = {name: load_json_file(path) for name, path in downloaded_paths.items()}

expected_labels = ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent',
                   'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt',
                   'RatioOfTheDecision', 'RulingByPresentCourt', 'None']

_lv = raw_data['label_vocab']
if isinstance(_lv, dict):
    if all(str(k).lstrip('-').isdigit() for k in _lv):          # {id: label}
        label_to_id = {str(v): int(k) for k, v in _lv.items()}
    else:                                                       # {label: id}
        label_to_id = {str(k): int(v) for k, v in _lv.items()}
else:
    label_to_id = {str(l): i for i, l in enumerate(_lv)}
id_to_label = [l for l, _ in sorted(label_to_id.items(), key=lambda kv: kv[1])]
NUM_LABELS = len(id_to_label)
ALL_IDS = list(range(NUM_LABELS))
assert NUM_LABELS == 13 and sorted(label_to_id.values()) == ALL_IDS and set(id_to_label) == set(expected_labels)

cl_train_docs = normalize_documents(raw_data['cl_train'], 'CL_train'); cl_dev_docs = normalize_documents(raw_data['cl_dev'], 'CL_dev')
it_train_docs = normalize_documents(raw_data['it_train'], 'IT_train'); it_dev_docs = normalize_documents(raw_data['it_dev'], 'IT_dev')
cl_test_docs  = normalize_documents(raw_data['cl_test'], 'CL_test');   it_test_docs = normalize_documents(raw_data['it_test'], 'IT_test')

def validate_labeled(docs, name):
    assert len({d['id'] for d in docs}) == len(docs), f'duplicate ids in {name}'
    for d in docs:
        assert isinstance(d.get('labels'), list) and len(d['labels']) == len(d['text']), f'{name}/{d["id"]}: sentence/label mismatch'
        assert set(d['labels']) <= set(expected_labels), f'{name}/{d["id"]}: unknown label'

def count_labels(docs):
    return Counter(l for d in docs for l in d['labels'])

for nm, docs in [('CL train', cl_train_docs), ('CL dev', cl_dev_docs), ('IT train', it_train_docs), ('IT dev', it_dev_docs)]:
    validate_labeled(docs, nm)
print(f'{"label":26s}{"CL train":>9s}{"CL dev":>8s}{"IT train":>9s}{"IT dev":>8s}')
_c = [count_labels(x) for x in (cl_train_docs, cl_dev_docs, it_train_docs, it_dev_docs)]
for l in expected_labels:
    print(f'{l:26s}' + ''.join(f'{c[l]:>{w}d}' for c, w in zip(_c, (9, 8, 9, 8))))
for nm, docs in [('CL train', cl_train_docs), ('CL dev', cl_dev_docs), ('IT train', it_train_docs), ('IT dev', it_dev_docs),
                 ('CL test', cl_test_docs), ('IT test', it_test_docs)]:
    print(f'{nm:9s} docs={len(docs):3d} sentences={sum(len(d["text"]) for d in docs):6d}')

label                      CL train  CL dev IT train  IT dev
Fact                           2979     249     1119     195
Issue                            17       3       90      12
ArgumentPetitioner              976     344      637      56
ArgumentRespondent              996     236      352      36
Statute                         367      58      732      62
Dissent                         284       0        0       0
PrecedentReliedUpon            1305     142      640     112
PrecedentNotReliedUpon          209       0      360      50
PrecedentOverruled                0       0        3       0
RulingByLowerCourt              438      56      436      46
RatioOfTheDecision             2856     288     1445     115
RulingByPresentCourt            113      18      194      31
None                             53       2      244      31
CL train  docs= 40 sentences= 10593
CL dev    docs=  5 sentences=  1396
IT train  docs= 40 sentences=  6252
IT dev    docs=  5 sentences=   746
CL

## 3. Phase 1 tokenizer (inference-only copy of the A1 BPE class — encodings are identical to Phase 1)

In [6]:
import heapq

SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
N_SPECIAL = 4
N_BYTES = 256
BASE_VOCAB_SIZE = N_SPECIAL + N_BYTES
WORD_RE = re.compile(r"\S+")


class A1BPETokenizer:
    def __init__(self, itos, id_to_bytes, merges):
        self.itos = list(itos)
        self._id_to_bytes = list(id_to_bytes)
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = len(self.itos)
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self._byte_to_id = {bytes([b]): N_SPECIAL + b for b in range(256)}
        self._merge_ranks = {pair: r for r, pair in enumerate(self.merges)}

    def _encode_word(self, word):
        raw = word.encode("utf-8")
        seq = [self._byte_to_id[bytes([b])] for b in raw]
        if len(seq) < 2 or not self._merge_ranks:
            return seq
        n = len(seq)
        prev = [-1] + list(range(0, n - 1))
        nxt = list(range(1, n)) + [-1]
        active = [True] * n
        heap = []
        for i in range(n - 1):
            rank = self._merge_ranks.get((seq[i], seq[i + 1]))
            if rank is not None:
                heapq.heappush(heap, (rank, i, i + 1))
        while heap:
            rank, left, right = heapq.heappop(heap)
            if not active[left] or not active[right]:
                continue
            if nxt[left] != right or prev[right] != left:
                continue
            if self._merge_ranks.get((seq[left], seq[right])) != rank:
                continue
            seq[left] = BASE_VOCAB_SIZE + rank
            active[right] = False
            right_next = nxt[right]
            nxt[left] = right_next
            if right_next != -1:
                prev[right_next] = left
            left_prev = prev[left]
            if left_prev != -1:
                new_rank = self._merge_ranks.get((seq[left_prev], seq[left]))
                if new_rank is not None:
                    heapq.heappush(heap, (new_rank, left_prev, left))
            if right_next != -1:
                new_rank = self._merge_ranks.get((seq[left], seq[right_next]))
                if new_rank is not None:
                    heapq.heappush(heap, (new_rank, left, right_next))
        out = []
        i = 0
        while i != -1:
            if active[i]:
                out.append(seq[i])
            i = nxt[i]
        return out

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError("text must be a string")
        ids = []
        cursor = 0
        for m in WORD_RE.finditer(text):
            if m.start() > cursor:
                ids.extend(N_SPECIAL + b for b in text[cursor:m.start()].encode("utf-8"))
            ids.extend(self._encode_word(m.group(0)))
            cursor = m.end()
        if cursor < len(text):
            ids.extend(N_SPECIAL + b for b in text[cursor:].encode("utf-8"))
        return ids


def load_a1_tokenizer(path, expected_vocab=50_000):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    itos = payload["itos"]
    merges = [tuple(pair) for pair in payload["merges"]]
    id_to_bytes = [b""] * N_SPECIAL + [bytes([b]) for b in range(N_BYTES)]
    for left_id, right_id in merges:
        id_to_bytes.append(id_to_bytes[left_id] + id_to_bytes[right_id])
    if len(itos) != len(id_to_bytes):
        raise ValueError(f"itos length={len(itos)}, id_to_bytes length={len(id_to_bytes)}")
    tok = A1BPETokenizer(itos, id_to_bytes, merges)
    if expected_vocab is not None and tok.vocab_size != expected_vocab:
        raise ValueError(f"Expected vocabulary size {expected_vocab}, got {tok.vocab_size}")
    assert list(tok.itos[:4]) == SPECIAL_TOKENS, f"unexpected special tokens: {tok.itos[:4]}"
    return tok

In [7]:
phase1_tokenizer = load_a1_tokenizer(str(PHASE1_TOKENIZER_PATH))
PAD_ID, UNK_ID, BOS_ID, EOS_ID = (phase1_tokenizer.stoi[t] for t in SPECIAL_TOKENS)
print('tokenizer vocab:', phase1_tokenizer.vocab_size, '| pad/unk/bos/eos ids:', PAD_ID, UNK_ID, BOS_ID, EOS_ID)
_probe = 'The petitioner filed the present appeal under Section 10 of the Companies Act, 2013.'
print('probe tokens:', len(phase1_tokenizer.encode(_probe)), '->', phase1_tokenizer.encode(_probe)[:12], '...')

tokenizer vocab: 50000 | pad/unk/bos/eos ids: 0 1 2 3
probe tokens: 27 -> [325, 36, 369, 36, 550, 36, 261, 36, 543, 36, 601, 36] ...


## 4. Phase 1 **encoder** — checkpoint and architecture
The classes below are the Phase 1 encoder (`BidirectionalGQAAttention`, `EncoderBlock`, `LegalEncoderForMLM`) with **identical parameter names**, plus a `key_mask` argument so padded batches are handled correctly (bidirectional attention would otherwise attend to `<pad>`; Phase 1 never needed this because every pre-training sequence was a full 512-token window). Weights are loaded with `strict=True`.

In [8]:
from dataclasses import dataclass, fields, asdict

_ck = torch.load(PHASE1_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
assert 'model_state_dict' in _ck and 'model_config' in _ck, f'unexpected checkpoint keys: {list(_ck)[:10]}'
if _ck.get('architecture') not in (None, 'bidirectional_gqa_encoder'):
    print('WARNING: checkpoint architecture is', _ck.get('architecture'), '- expected bidirectional_gqa_encoder. '
          'Is this the decoder (GQA/SWA) checkpoint by mistake?')
phase1_state = _ck['model_state_dict']
model_config = dict(_ck['model_config'])
MASK_ID = int(_ck.get('mask_token_id', UNK_ID))        # Phase 1 used <unk> as the MLM corruption sentinel
print('checkpoint objective :', _ck.get('pretraining_objective'), '| architecture:', _ck.get('architecture'))
print('best step / val MLM  :', _ck.get('global_step'), '/', _ck.get('best_val_mlm_loss'))
print('mask sentinel id     :', MASK_ID, '| config:', model_config)
del _ck; gc.collect()


@dataclass
class ModelConfig:
    vocab_size: int = 50_000
    d_model: int = 768
    n_layers: int = 16
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048
    max_seq_len: int = 512
    rope_base: float = 10_000.0
    rms_norm_eps: float = 1e-6
    dropout: float = 0.0

cfg = ModelConfig(**{k: v for k, v in model_config.items() if k in {f.name for f in fields(ModelConfig)}})
assert cfg.vocab_size == phase1_tokenizer.vocab_size, 'tokenizer / checkpoint vocabulary mismatch'
assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
print(cfg)

checkpoint objective : masked_language_modeling | architecture: bidirectional_gqa_encoder
best step / val MLM  : 15000 / 1.0438882003729564
mask sentinel id     : 1 | config: {'vocab_size': 50000, 'd_model': 768, 'n_layers': 16, 'n_heads': 12, 'n_kv_heads': 4, 'd_ff': 2048, 'max_seq_len': 512, 'rope_base': 10000.0, 'rms_norm_eps': 1e-06, 'dropout': 0.0}
ModelConfig(vocab_size=50000, d_model=768, n_layers=16, n_heads=12, n_kv_heads=4, d_ff=2048, max_seq_len=512, rope_base=10000.0, rms_norm_eps=1e-06, dropout=0.0)


In [9]:
def build_rope_cache(max_seq_len, d_head, base=10_000.0):
    assert d_head % 2 == 0
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inv_freq)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.to(device=x.device, dtype=x.dtype).unsqueeze(0).unsqueeze(0)
    sin = sin.to(device=x.device, dtype=x.dtype).unsqueeze(0).unsqueeze(0)
    return x * cos + rotate_half(x) * sin


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        input_dtype = x.dtype
        xf = x.float()
        norm = torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (xf * norm * self.weight.float()).to(input_dtype)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class BidirectionalGQAAttention(nn.Module):
    """Phase 1 attention (same parameter names) + optional key padding mask. key_mask: [B, T] bool, True = real token."""
    def __init__(self, config):
        super().__init__()
        self.d_model, self.n_heads, self.n_kv_heads = config.d_model, config.n_heads, config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = config.n_heads // config.n_kv_heads
        self.max_seq_len = config.max_seq_len
        self.q_proj = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        cos, sin = build_rope_cache(config.max_seq_len, self.head_dim, config.rope_base)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

    def _split_heads(self, x, n_heads):
        B, T, _ = x.shape
        return x.reshape(B, T, n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, key_mask=None):
        B, T, _ = x.shape
        if T > self.max_seq_len:
            raise ValueError(f'sequence length {T} exceeds {self.max_seq_len}')
        q = self._split_heads(self.q_proj(x), self.n_heads)
        k = self._split_heads(self.k_proj(x), self.n_kv_heads)
        v = self._split_heads(self.v_proj(x), self.n_kv_heads)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        mask = None if key_mask is None else key_mask[:, None, None, :]          # [B,1,1,T], broadcast over heads / queries
        attn = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
        attn = attn.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.out_proj(attn)


class EncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.attn = BidirectionalGQAAttention(config)
        self.ffn_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.mlp = SwiGLU(config.d_model, config.d_ff)

    def forward(self, x, key_mask=None):
        x = x + self.attn(self.attn_norm(x), key_mask)
        x = x + self.mlp(self.ffn_norm(x))
        return x


class LegalEncoder(nn.Module):
    """Phase 1 encoder without the (tied) MLM head. forward() -> final hidden states [B, T, d_model]."""
    def __init__(self, config, grad_ckpt=False):
        super().__init__()
        self.config = config
        self.grad_ckpt = grad_ckpt
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.layers = nn.ModuleList([EncoderBlock(config) for _ in range(config.n_layers)])
        self.final_norm = RMSNorm(config.d_model, config.rms_norm_eps)

    def forward(self, input_ids, key_mask=None):
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(layer, x, key_mask, use_reentrant=False)
            else:
                x = layer(x, key_mask)
        return self.final_norm(x)

    def mlm_logits(self, hidden):                       # tied head, as in Phase 1
        return F.linear(hidden, self.token_embedding.weight)


def load_phase1_encoder_weights(encoder, state):
    sd = {}
    for k, v in state.items():
        for pre in ('module.', '_orig_mod.'):
            if k.startswith(pre):
                k = k[len(pre):]
        sd[k] = v
    tied = sd.pop('mlm_head.weight', None)               # tied to token_embedding in Phase 1
    if tied is not None and not torch.equal(tied.cpu(), sd['token_embedding.weight'].cpu()):
        print('WARNING: mlm_head.weight differs from token_embedding.weight (not tied?) - ignoring mlm_head.')
    encoder.load_state_dict(sd, strict=True)             # raises on any missing / unexpected key
    return encoder

_t = LegalEncoder(cfg)
load_phase1_encoder_weights(_t, phase1_state)
print('Phase 1 encoder weights loaded (strict). Parameters:', f'{sum(p.numel() for p in _t.parameters()):,}')
del _t; gc.collect()

Phase 1 encoder weights loaded (strict). Parameters: 139,088,640


0

## 5. LoRA (self-contained)
`W x + (alpha/r)·B(A(dropout(x)))`, `A` Kaiming-initialised, `B` zero-initialised (so the adapted model starts *exactly* at the Phase 1 model). `rslora=True` uses `alpha/sqrt(r)` (rank-stabilised LoRA). Adapters are injected into every `q/k/v/out/gate/up/down` projection; the Phase 1 weights stay frozen.

In [10]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0, rslora=False):
        super().__init__()
        assert isinstance(base, nn.Linear) and r > 0
        self.base = base
        self.r = r
        self.scaling = (alpha / math.sqrt(r)) if rslora else (alpha / r)
        self.lora_dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lora_A = nn.Linear(base.in_features, r, bias=False)       # default init = Kaiming uniform(a=sqrt(5))
        self.lora_B = nn.Linear(r, base.out_features, bias=False)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.base(x) + self.lora_B(self.lora_A(self.lora_dropout(x))) * self.scaling


def inject_lora(model, target_names, r, alpha, dropout=0.0, rslora=False):
    """Wrap every nn.Linear child whose attribute name is in target_names. Returns the number of wrapped layers."""
    target_names = set(target_names)
    n = 0
    for parent in list(model.modules()):
        for child_name, child in list(parent.named_children()):
            if child_name in target_names and isinstance(child, nn.Linear):
                setattr(parent, child_name, LoRALinear(child, r, alpha, dropout, rslora))
                n += 1
    return n

print('LoRA ready.')

LoRA ready.


## 6. Why a hierarchical model? (label runs and position — unchanged analysis)
Labels come in long runs (P(label_i == label_(i-1)) ≈ 0.87) and are tied to where a sentence sits in the judgment, so the model gets a document-level BiLSTM over sentence vectors plus position features.

In [11]:
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, accuracy_score
from contextlib import nullcontext

device = torch.device('cuda')
USE_AMP = True
AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # bf16 on Ampere+ (A100/L4/H100); fp16 + GradScaler on T4
USE_GRAD_SCALER = AMP_DTYPE == torch.float16

def make_autocast():
    return torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=True) if USE_AMP else nullcontext()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print('device:', device, '| AMP dtype:', AMP_DTYPE)

rows, same_prev, total_pairs = [], 0, 0
for dom, docs in [('CL', cl_train_docs), ('IT', it_train_docs)]:
    for d in docs:
        n = len(d['labels'])
        for i, l in enumerate(d['labels']):
            rows.append((dom, l, i / max(n - 1, 1)))
            if i > 0:
                total_pairs += 1
                same_prev += int(l == d['labels'][i - 1])
pos_df = pd.DataFrame(rows, columns=['domain', 'label', 'rel_pos'])
display(pos_df.groupby(['domain', 'label']).rel_pos.agg(['count', 'mean', 'std']).round(3))
print(f'P(label_i == label_(i-1)) = {same_prev / total_pairs:.3f}')

device: cuda | AMP dtype: torch.bfloat16


count   mean    std
domain label                                      
CL     ArgumentPetitioner        976  0.326  0.188
       ArgumentRespondent        996  0.388  0.161
       Dissent                   284  0.872  0.083
       Fact                     2979  0.294  0.262
       Issue                      17  0.348  0.208
       None                       53  0.765  0.230
       PrecedentNotReliedUpon    209  0.317  0.229
       PrecedentReliedUpon      1305  0.706  0.211
       RatioOfTheDecision       2856  0.682  0.194
       RulingByLowerCourt        438  0.487  0.294
       RulingByPresentCourt      113  0.966  0.064
       Statute                   367  0.440  0.179
IT     ArgumentPetitioner        637  0.404  0.243
       ArgumentRespondent        352  0.442  0.226
       Fact                     1119  0.264  0.258
       Issue                      90  0.357  0.285
       None                      244  0.456  0.357
       PrecedentNotReliedUpon    360  0.617  0.204
       PrecedentOverruled          3  0.599  0.033
       PrecedentReliedUpon       640  0.606  0.192
       RatioOfTheDecision       1445  0.670  0.228
       RulingByLowerCourt        436  0.397  0.288
       RulingByPresentCourt      194  0.931  0.124
       Statute                   732  0.466  0.233

P(label_i == label_(i-1)) = 0.869


## 7. Tokenise once; build the encoder input for each sentence
Every sentence is tokenised once. The encoder input for sentence *i* is
`<s> [left context] TARGET </s> [right context]` where context sentences (separated by `</s>`) are only added when `ctx_k > 0`; the sentence vector is pooled **only over the TARGET tokens**. With `ctx_k=0` this is just `<s> TARGET </s>` (one sentence per row, as in the GQA v2 notebook). Context is capped at 64 tokens per side (left context keeps its *last* tokens, right context its *first*), the target at 126 tokens, so inputs are ≤ 256 tokens (the model was pre-trained at 512).

In [12]:
MAX_TGT_TOKENS = 126      # sentence content tokens (median ~56, p95 ~160 in the GQA notebook; longer sentences keep their first 126)
CTX_BUDGET = 64           # max context tokens per side when ctx_k > 0

def build_records(docs, domain, has_labels=True):
    recs = []
    for d in docs:
        toks = []
        for s in d['text']:
            t = phase1_tokenizer.encode(s.strip())
            if not t:
                t = phase1_tokenizer.encode('.')
            toks.append(t[:MAX_TGT_TOKENS])
        labels = None
        if has_labels:
            labels = np.array([label_to_id[l] for l in d['labels']], dtype=np.int64)
            assert len(labels) == len(toks)
        recs.append(dict(doc_id=str(d['id']), domain=domain, domain_id=0 if domain == 'CL' else 1,
                         tok=toks, labels=labels, n=len(toks)))
    return recs


def make_input(rec, i, ctx_k):
    """-> (token_ids, target_start, target_end). Only [target_start, target_end) is pooled."""
    tgt = rec['tok'][i]
    left, right = [], []
    if ctx_k > 0:
        n = rec['n']
        for j in range(max(0, i - ctx_k), i):
            left += rec['tok'][j] + [EOS_ID]
        for j in range(i + 1, min(n, i + 1 + ctx_k)):
            right += rec['tok'][j] + [EOS_ID]
        left = left[-CTX_BUDGET:]
        right = right[:CTX_BUDGET]
    ids = [BOS_ID] + left + tgt + [EOS_ID] + right
    ts = 1 + len(left)
    return ids, ts, ts + len(tgt)


REC = {
    'cl_train': build_records(cl_train_docs, 'CL'), 'cl_dev': build_records(cl_dev_docs, 'CL'),
    'it_train': build_records(it_train_docs, 'IT'), 'it_dev': build_records(it_dev_docs, 'IT'),
    'cl_test': build_records(cl_test_docs, 'CL', False), 'it_test': build_records(it_test_docs, 'IT', False),
}
for k, v in REC.items():
    print(f'{k:9s} docs={len(v):3d} sentences={sum(r["n"] for r in v):6d}')
TRAIN_RECS = REC['cl_train'] + REC['it_train']
DEV_SETS = {'CL': REC['cl_dev'], 'IT': REC['it_dev']}
_lens = [len(t) for r in TRAIN_RECS for t in r['tok']]
print('train sentence length (tokens): mean %.1f | median %d | p95 %d | truncated at %d: %.2f%%' % (
    np.mean(_lens), np.median(_lens), np.percentile(_lens, 95), MAX_TGT_TOKENS, 100 * np.mean(np.array(_lens) >= MAX_TGT_TOKENS)))
_ex_ids, _ts, _te = make_input(TRAIN_RECS[0], 3, 1)
print('example input (ctx_k=1): len', len(_ex_ids), '| target span', (_ts, _te), '| BOS first:', _ex_ids[0] == BOS_ID)

cl_train  docs= 40 sentences= 10593
cl_dev    docs=  5 sentences=  1396
it_train  docs= 40 sentences=  6252
it_dev    docs=  5 sentences=   746
cl_test   docs=  5 sentences=  1339
it_test   docs=  5 sentences=   858
train sentence length (tokens): mean 62.6 | median 56 | p95 126 | truncated at 126: 10.06%
example input (ctx_k=1): len 138 | target span (53, 76) | BOS first: True


## 8. Windows, metrics, logit adjustment (pure numpy — identical to the GQA v2 notebook)
Training: non-overlapping 32-sentence windows with a random offset per epoch. Inference: overlapping windows (stride 16) with triangular weighting. Metrics: scikit-learn default macro-F1 (most likely leaderboard definition), plus the old 13-label number for comparison.

In [13]:
NB = 24   # relative-position buckets

def make_train_windows(records, window, rng):
    wins = []
    for ri, r in enumerate(records):
        n = r['n']
        off = int(rng.integers(0, window)) if window > 1 else 0
        for s in range(-off, n, window):
            a, b = max(s, 0), min(s + window, n)
            if b > a:
                wins.append((ri, a, b))
    return wins

def make_eval_windows(records, window, stride):
    wins = []
    for ri, r in enumerate(records):
        n = r['n']
        if n <= window:
            wins.append((ri, 0, n)); continue
        starts = list(range(0, n - window + 1, stride))
        if starts[-1] + window < n:
            starts.append(n - window)
        wins.extend((ri, s, s + window) for s in starts)
    return wins

def win_weights(a, b, n):
    L = b - a
    j = np.arange(L)
    left = (j + 1) if a > 0 else np.full(L, 10 ** 6)
    right = (L - j) if b < n else np.full(L, 10 ** 6)
    return np.minimum(left, right).astype(np.float64)

def pack_batches(wins, budget, rng=None, shuffle=True):
    wins = list(wins)
    if shuffle:
        wins = [wins[i] for i in rng.permutation(len(wins))]
    batches, cur, cnt = [], [], 0
    for w in wins:
        L = w[2] - w[1]
        if cur and cnt + L > budget:
            batches.append(cur); cur, cnt = [], 0
        cur.append(w); cnt += L
    if cur:
        batches.append(cur)
    return batches

def rr_metrics(y_true, y_pred):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=ALL_IDS, average=None, zero_division=0)
    present = s > 0
    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_precision': float(precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)[0]),
        'macro_recall': float(precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)[1]),
        'macro_f1_present': float(f[present].mean()),
        'macro_f1_13': float(f.mean()),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'per_class': [dict(label=id_to_label[i], precision=float(p[i]), recall=float(r[i]), f1=float(f[i]), support=int(s[i]))
                      for i in ALL_IDS],
    }

def domain_metrics(logp_by_dom, dev_sets, tau=0.0, log_prior=None):
    out = {}
    for dom, recs in dev_sets.items():
        lp = np.concatenate(logp_by_dom[dom])
        if tau and log_prior is not None:
            lp = lp - tau * log_prior
        y_true = np.concatenate([r['labels'] for r in recs])
        out[dom] = rr_metrics(y_true, lp.argmax(1))
    return out

def mean_f1(metrics_by_dom, key='macro_f1'):
    return float(np.mean([m[key] for m in metrics_by_dom.values()]))

def train_log_prior(records):
    cnt = np.bincount(np.concatenate([r['labels'] for r in records]), minlength=NUM_LABELS).astype(np.float64)
    return np.log((cnt + 1.0) / (cnt.sum() + NUM_LABELS))

def tune_tau(logp_by_dom, dev_sets, log_prior, grid=(0.0, 0.25, 0.5, 0.75, 1.0, 1.25)):
    rows = []
    for tau in grid:
        m = domain_metrics(logp_by_dom, dev_sets, tau, log_prior)
        rows.append(dict(tau=tau, CL=m['CL']['macro_f1'], IT=m['IT']['macro_f1'], mean=mean_f1(m)))
    df = pd.DataFrame(rows)
    best = df['mean'].max()
    tau = float(df[df['mean'] >= best - 0.003].sort_values('tau').iloc[0]['tau'])   # smallest tau within 0.003 of the best
    return tau, df

def class_weights_from(records, power, clip=(0.3, 3.0)):
    if power <= 0:
        return None
    cnt = np.bincount(np.concatenate([r['labels'] for r in records]), minlength=NUM_LABELS).astype(np.float64)
    w = np.ones(NUM_LABELS); pres = cnt > 0
    w[pres] = (cnt[pres].max() / cnt[pres]) ** power
    w = np.clip(w, *clip)
    w /= (w[pres] * cnt[pres]).sum() / cnt[pres].sum()
    return w.astype(np.float32)

print('Window / metric utilities ready.')

Window / metric utilities ready.


## 9. Model
`HierEncRRModel` = **Phase 1 encoder** (frozen; LoRA adapters) → per-sentence vector `[mean over target tokens ; attention-pooled target tokens]` → + domain + position features → **2-layer BiLSTM over the sentences of a window** → LayerNorm → 13-way classifier (exactly one label per sentence). `lora_r=0` gives a fully frozen encoder (linear-probe style control, experiment E5).

In [14]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence


class HierEncRRModel(nn.Module):
    def __init__(self, encoder, n_labels=NUM_LABELS, proj_dim=512, lstm_hidden=256, lstm_layers=2, dropout=0.3,
                 use_doc_context=True, use_position=True, ctx_k=0, backbone_trainable=True, enc_chunk=64):
        super().__init__()
        self.encoder = encoder
        self.use_doc_context, self.use_position = use_doc_context, use_position
        self.ctx_k, self.backbone_trainable, self.enc_chunk = ctx_k, backbone_trainable, enc_chunk
        d = encoder.config.d_model
        self.attn_pool = nn.Linear(d, 1)                       # learned attention pooling over the target tokens
        self.sent_proj = nn.Sequential(nn.Linear(2 * d, proj_dim), nn.GELU(), nn.Dropout(dropout))
        self.dom_emb = nn.Embedding(2, 16)
        self.pos_bucket = nn.Embedding(NB, 32)
        extra = 16 + ((3 + 32) if use_position else 0)
        self.in_proj = nn.Linear(proj_dim + extra, proj_dim)
        ctx_dim = 2 * lstm_hidden
        if use_doc_context:
            self.lstm = nn.LSTM(proj_dim, lstm_hidden, num_layers=lstm_layers, batch_first=True, bidirectional=True,
                                dropout=dropout if lstm_layers > 1 else 0.0)
        else:                                                   # sentence-only control with a comparable head size
            self.mlp_ctx = nn.Sequential(nn.Linear(proj_dim, ctx_dim), nn.GELU())
        self.out_norm = nn.LayerNorm(proj_dim + ctx_dim)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(proj_dim + ctx_dim, n_labels))

    def encode_sentences(self, inputs):
        """inputs: list of (ids, target_start, target_end). Length-sorted chunks -> little padding.
        Returns [N, 2d] = [mean over target tokens ; attention-pooled target tokens], in the original order."""
        dev = next(self.parameters()).device
        n = len(inputs)
        order = sorted(range(n), key=lambda i: len(inputs[i][0]))
        feats = []
        for s in range(0, n, self.enc_chunk):
            idx = order[s:s + self.enc_chunk]
            lens = [len(inputs[i][0]) for i in idx]
            T = max(lens)
            ids = torch.full((len(idx), T), PAD_ID, dtype=torch.long)
            for r_, i in enumerate(idx):
                ids[r_, :lens[r_]] = torch.tensor(inputs[i][0], dtype=torch.long)
            ids = ids.to(dev)
            lens_t = torch.tensor(lens, device=dev)
            ts = torch.tensor([inputs[i][1] for i in idx], device=dev)
            te = torch.tensor([inputs[i][2] for i in idx], device=dev)
            pos = torch.arange(T, device=dev)[None, :]
            key_mask = pos < lens_t[:, None]                    # real tokens (bidirectional attention must not see <pad>)
            span = (pos >= ts[:, None]) & (pos < te[:, None])   # target-sentence tokens (always non-empty)
            if self.backbone_trainable:
                hidden = self.encoder(ids, key_mask)
            else:
                with torch.no_grad():
                    hidden = self.encoder(ids, key_mask)
            hidden = hidden.float()
            spanf = span.unsqueeze(-1).float()
            mean = (hidden * spanf).sum(1) / spanf.sum(1).clamp(min=1)
            score = self.attn_pool(hidden).squeeze(-1).float().masked_fill(~span, float('-inf'))
            attn = (torch.softmax(score, dim=-1).unsqueeze(-1) * hidden).sum(1)
            feats.append(torch.cat([mean, attn], dim=-1))
        feats = torch.cat(feats, 0)
        inv = torch.empty(n, dtype=torch.long, device=dev)
        inv[torch.tensor(order, device=dev)] = torch.arange(n, device=dev)
        return feats[inv]

    def forward(self, inputs, win_lens, domain_ids, pos_feats, pos_bucket):
        x = self.sent_proj(self.encode_sentences(inputs))
        parts = [x, self.dom_emb(domain_ids)]
        if self.use_position:
            parts += [pos_feats.to(x.dtype), self.pos_bucket(pos_bucket)]
        h = F.gelu(self.in_proj(torch.cat(parts, dim=-1)))
        if self.use_doc_context:
            padded = pad_sequence(list(torch.split(h, win_lens)), batch_first=True)
            packed = pack_padded_sequence(padded, torch.tensor(win_lens), batch_first=True, enforce_sorted=False)
            out, _ = self.lstm(packed)
            out, _ = pad_packed_sequence(out, batch_first=True, total_length=padded.size(1))
            ctx = torch.cat([out[b, :L] for b, L in enumerate(win_lens)], dim=0).to(h.dtype)
        else:
            ctx = self.mlp_ctx(h)
        return self.classifier(self.out_norm(torch.cat([h, ctx], dim=-1)))


def collate_windows(records, wins, ctx_k):
    inputs, win_lens, dom, posf, bucket, labels = [], [], [], [], [], []
    for ri, a, b in wins:
        r = records[ri]; n = r['n']
        for i in range(a, b):
            inputs.append(make_input(r, i, ctx_k))
            dom.append(r['domain_id'])
            rel = i / max(n - 1, 1)
            posf.append([rel, math.log1p(i) / 7.0, math.log1p(n - 1 - i) / 7.0])
            bucket.append(min(int(rel * NB), NB - 1))
            if r['labels'] is not None:
                labels.append(int(r['labels'][i]))
        win_lens.append(b - a)
    return dict(inputs=inputs, win_lens=win_lens,
                domain_ids=torch.tensor(dom, dtype=torch.long),
                pos_feats=torch.tensor(posf, dtype=torch.float32),
                pos_bucket=torch.tensor(bucket, dtype=torch.long),
                labels=torch.tensor(labels, dtype=torch.long) if labels else None)


def model_forward(model, batch):
    return model(batch['inputs'], batch['win_lens'], batch['domain_ids'].to(device),
                 batch['pos_feats'].to(device), batch['pos_bucket'].to(device))


@torch.inference_mode()
def predict_logp(model, records, window, stride, batch_sentences=128):
    """Weighted average of per-window log-probabilities -> list of [n_i, 13] arrays (one per document)."""
    model.eval()
    acc = [np.zeros((r['n'], NUM_LABELS)) for r in records]
    wsum = [np.zeros(r['n']) for r in records]
    for bw in pack_batches(make_eval_windows(records, window, stride), batch_sentences, shuffle=False):
        batch = collate_windows(records, bw, model.ctx_k)
        with make_autocast():
            logits = model_forward(model, batch)
        logp = F.log_softmax(logits.float(), dim=-1).cpu().numpy()
        k = 0
        for ri, a, b in bw:
            L = b - a
            w = win_weights(a, b, records[ri]['n'])
            acc[ri][a:b] += logp[k:k + L] * w[:, None]
            wsum[ri][a:b] += w
            k += L
    return [acc[i] / wsum[i][:, None] for i in range(len(records))]

print('HierEncRRModel ready.')

HierEncRRModel ready.


## 10. Building a fresh model and the training loop (warm-up + cosine LR, class weights, label smoothing, per-epoch dev evaluation)

In [15]:
def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, trn


def build_model(hp, seed):
    """Fresh Phase 1 encoder -> frozen -> (optional) LoRA -> hierarchical RR head."""
    seed_everything(seed)
    enc = LegalEncoder(cfg, grad_ckpt=hp['grad_ckpt'])
    load_phase1_encoder_weights(enc, phase1_state)
    for p in enc.parameters():
        p.requires_grad_(False)
    n_wrapped = 0
    if hp['lora_r'] > 0:
        n_wrapped = inject_lora(enc, hp['target_modules'], hp['lora_r'], hp['lora_alpha'], hp['lora_dropout'], hp['rslora'])
    model = HierEncRRModel(enc, proj_dim=hp['proj_dim'], lstm_hidden=hp['lstm_hidden'], lstm_layers=hp['lstm_layers'],
                           dropout=hp['dropout'], use_doc_context=hp['use_doc_context'], use_position=hp['use_position'],
                           ctx_k=hp['ctx_k'], backbone_trainable=hp['lora_r'] > 0, enc_chunk=hp['enc_chunk'])
    model.n_lora_layers = n_wrapped
    return model.to(device)


def trainable_state(model):
    sd = model.state_dict()
    return {n: sd[n].detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def fit_model(hp, train_recs, dev_sets=None, seed=42, tag=''):
    """Train a fresh model. Returns the model at the LAST epoch plus bookkeeping (best epoch by mean dev macro-F1)."""
    seed_everything(seed)
    rng = np.random.default_rng(seed)
    model = build_model(hp, seed)
    total_p, train_p = count_params(model)

    lora_params = [p for n, p in model.encoder.named_parameters() if p.requires_grad]
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('encoder.')]
    groups = ([{'params': lora_params, 'lr': hp['lora_lr']}] if lora_params else []) + [{'params': head_params, 'lr': hp['head_lr']}]
    opt = torch.optim.AdamW(groups, weight_decay=hp['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)

    epoch_batches = [pack_batches(make_train_windows(train_recs, hp['window'], rng), hp['batch_sentences'], rng, True)
                     for _ in range(hp['epochs'])]
    total_steps = sum(len(b) for b in epoch_batches)
    warm = max(1, int(hp['warmup'] * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))

    cw = class_weights_from(train_recs, hp['cw_power'])
    cw_t = None if cw is None else torch.tensor(cw, device=device)
    print(f'[{tag}] params total={total_p:,} trainable={train_p:,} ({100 * train_p / total_p:.2f}%) | LoRA layers={model.n_lora_layers} | '
          f'steps={total_steps} | class weights={"on" if cw is not None else "off"}')

    history, best = [], dict(score=-1.0, epoch=None, state=None, logp=None, metrics=None)
    last_logp = None
    t_start = time.time()
    torch.cuda.reset_peak_memory_stats()

    for ep in range(1, hp['epochs'] + 1):
        t0 = time.time(); model.train(); run_loss, run_n = 0.0, 0
        for bw in epoch_batches[ep - 1]:
            batch = collate_windows(train_recs, bw, model.ctx_k)
            labels = batch['labels'].to(device)
            with make_autocast():
                logits = model_forward(model, batch)
            loss = F.cross_entropy(logits.float(), labels, weight=cw_t, label_smoothing=hp['label_smoothing'])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(lora_params + head_params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += float(loss.detach()) * len(labels); run_n += len(labels)

        row = dict(epoch=ep, train_loss=run_loss / max(run_n, 1), train_seconds=time.time() - t0)
        if dev_sets:
            logp = {d: predict_logp(model, recs, hp['window'], hp['eval_stride']) for d, recs in dev_sets.items()}
            m = domain_metrics(logp, dev_sets)
            row.update(cl_macro_f1=m['CL']['macro_f1'], it_macro_f1=m['IT']['macro_f1'], mean_macro_f1=mean_f1(m),
                       cl_macro_f1_13=m['CL']['macro_f1_13'], it_macro_f1_13=m['IT']['macro_f1_13'],
                       cl_acc=m['CL']['accuracy'], it_acc=m['IT']['accuracy'])
            last_logp = logp
            if row['mean_macro_f1'] > best['score']:
                best.update(score=row['mean_macro_f1'], epoch=ep, state=trainable_state(model), logp=logp, metrics=m)
        history.append(row)
        print(f'[{tag}] ' + ' | '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in row.items()))

    info = dict(history=history, best_epoch=best['epoch'], best_score=best['score'], best_state=best['state'],
                best_logp=best['logp'], best_metrics=best['metrics'], last_logp=last_logp,
                total_params=int(total_p), trainable_params=int(train_p),
                minutes=(time.time() - t_start) / 60, peak_gb=torch.cuda.max_memory_allocated() / 1024 ** 3)
    return model, info


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

print('Training loop ready.')

Training loop ready.


## 11. Experiments — configuration
A controlled ladder (identical budget, seed 42) answering *does context help?*, *does the encoder's bidirectionality help?*, *is the Phase 1 representation useful without adaptation?* and *what does each ingredient buy?*

| Run | Document BiLSTM | Position | Neighbours inside the encoder | Class weights | LoRA | Window |
|---|---|---|---|---|---|---|
| `E0_sentence_only` | – | – | – | – | r16 all-linear | – |
| `E1_hier` | ✔ | ✔ | – | – | r16 | 32 |
| `E2_hier_cw` | ✔ | ✔ | – | ✔ | r16 | 32 |
| `E3_tokctx1` | ✔ | ✔ | ±1 sentence | – | r16 | 32 |
| `E4_tokctx1_cw` | ✔ | ✔ | ±1 sentence | ✔ | r16 | 32 |
| `E5_frozen_probe` | ✔ | ✔ | – | – | **none (encoder frozen)** | 32 |
| `E6_win64_cw` | ✔ | ✔ | – | ✔ | r16 | 64 |

For the write-up: E0→E1 = effect of document-level context; E1→E3 = effect of bidirectional token-level context; E1 vs E5 = how much LoRA adaptation adds to the raw Phase 1 representation.

In [16]:
BASE_HP = dict(
    window=32, eval_stride=16, batch_sentences=128,
    use_doc_context=True, use_position=True, ctx_k=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj', 'gate_proj', 'up_proj', 'down_proj'],   # all linear layers
    lora_r=16, lora_alpha=32, lora_dropout=0.1, rslora=False,
    lora_lr=3e-4, head_lr=1e-3, weight_decay=0.01, warmup=0.06, epochs=8,
    proj_dim=512, lstm_hidden=256, lstm_layers=2, dropout=0.3,
    label_smoothing=0.05, cw_power=0.0,
    grad_ckpt=True,      # set False for speed if the GPU has plenty of memory (A100/H100-class)
    enc_chunk=64,
)
EXPERIMENTS = {
    'E0_sentence_only': dict(use_doc_context=False, use_position=False, eval_stride=32),
    'E1_hier':          dict(),
    'E2_hier_cw':       dict(cw_power=0.5),
    'E3_tokctx1':       dict(ctx_k=1),
    'E4_tokctx1_cw':    dict(ctx_k=1, cw_power=0.5),
    'E5_frozen_probe':  dict(lora_r=0),
    'E6_win64_cw':      dict(window=64, eval_stride=32, cw_power=0.5),
}
EXPERIMENTS = {k: {**BASE_HP, **v} for k, v in EXPERIMENTS.items()}
RUN_ONLY = None          # e.g. ['E1_hier', 'E3_tokctx1'] to run a subset while debugging
SEED = 42
print({k: {kk: vv for kk, vv in v.items() if BASE_HP.get(kk) != vv} for k, v in EXPERIMENTS.items()})

{'E0_sentence_only': {'eval_stride': 32, 'use_doc_context': False, 'use_position': False}, 'E1_hier': {}, 'E2_hier_cw': {'cw_power': 0.5}, 'E3_tokctx1': {'ctx_k': 1}, 'E4_tokctx1_cw': {'ctx_k': 1, 'cw_power': 0.5}, 'E5_frozen_probe': {'lora_r': 0}, 'E6_win64_cw': {'window': 64, 'eval_stride': 32, 'cw_power': 0.5}}


## 12. Fail-fast checks (run before any training)
1. **Checkpoint sanity** — masked-LM loss of the loaded Phase 1 encoder on real RR sentences. Random guessing is ln(50000) ≈ 10.8; a correctly loaded encoder is far lower. If this is high, the checkpoint/tokenizer pairing is wrong and nothing downstream can work.
2. **Architecture checks** — LoRA layer count, frozen base weights, *padding invariance* (a padded batch must give the same sentence vectors as encoding each sentence alone), LoRA = identity at initialisation, and a forward/backward pass for the context, frozen-probe and standard variants.

In [17]:
@torch.no_grad()
def mlm_sanity(n_sent=256, bs=64, seed=0):
    rng = np.random.default_rng(seed)
    g = torch.Generator().manual_seed(seed)
    enc = LegalEncoder(cfg).to(device)
    load_phase1_encoder_weights(enc, phase1_state)
    enc.eval()
    cands = [t for r in TRAIN_RECS for t in r['tok'] if 8 <= len(t) <= 100]
    pick = rng.choice(len(cands), size=min(n_sent, len(cands)), replace=False)
    sents = [[BOS_ID] + cands[i] + [EOS_ID] for i in pick]
    tot_loss, tot_n, tot_ok = 0.0, 0, 0
    for s in range(0, len(sents), bs):
        chunk = sents[s:s + bs]
        T = max(len(x) for x in chunk)
        ids = torch.full((len(chunk), T), PAD_ID, dtype=torch.long)
        for i, x in enumerate(chunk):
            ids[i, :len(x)] = torch.tensor(x)
        ids = ids.to(device)
        key_mask = ids.ne(PAD_ID)
        eligible = key_mask & (ids >= 4)
        sel = (torch.rand(ids.shape, generator=g).to(device) < 0.15) & eligible
        for i in range(ids.size(0)):
            if not sel[i].any():
                sel[i, torch.nonzero(eligible[i]).flatten()[0]] = True
        draw = torch.rand(ids.shape, generator=g).to(device)
        corrupted = ids.clone()
        corrupted[sel & (draw < 0.8)] = MASK_ID
        rpos = sel & (draw >= 0.8) & (draw < 0.9)
        corrupted[rpos] = torch.randint(4, cfg.vocab_size, ids.shape, generator=g).to(device)[rpos]
        logits = enc.mlm_logits(enc(corrupted, key_mask)[sel]).float()
        target = ids[sel]
        tot_loss += F.cross_entropy(logits, target, reduction='sum').item()
        tot_ok += (logits.argmax(-1) == target).sum().item()
        tot_n += target.numel()
    del enc; free_gpu()
    return tot_loss / tot_n, tot_ok / tot_n, tot_n

_loss, _acc, _n = mlm_sanity()
print(f'Phase 1 encoder on {_n} masked tokens from RR sentences: MLM loss {_loss:.3f} (random = {math.log(cfg.vocab_size):.2f}) | top-1 acc {_acc:.3f}')
if _loss > 8.0:
    print('WARNING: loss is close to random. The checkpoint is probably not the trained encoder, or the tokenizer does not match it.')
elif _loss > 6.0:
    print('NOTE: loss is much better than random but high; sentences are short (little context), so this is plausible - compare with your Phase 1 validation MLM loss.')
else:
    print('OK: the Phase 1 encoder and tokenizer are consistent.')

Phase 1 encoder on 1955 masked tokens from RR sentences: MLM loss 1.736 (random = 10.82) | top-1 acc 0.722
OK: the Phase 1 encoder and tokenizer are consistent.


In [18]:
# --- 12.2 architecture smoke tests (a few seconds) ---
def _grad_sums(m):
    g_enc = sum(p.grad.abs().sum().item() for n, p in m.encoder.named_parameters() if p.grad is not None)
    g_head = sum(p.grad.abs().sum().item() for n, p in m.named_parameters() if p.grad is not None and not n.startswith('encoder.'))
    return g_enc, g_head

_rng = np.random.default_rng(0)
for _name in ['E1_hier', 'E3_tokctx1', 'E5_frozen_probe']:
    _hp = EXPERIMENTS[_name]
    _m = build_model(_hp, 0)
    tot, trn = count_params(_m)
    exp_wrapped = cfg.n_layers * len(_hp['target_modules']) if _hp['lora_r'] > 0 else 0
    assert _m.n_lora_layers == exp_wrapped, (_m.n_lora_layers, exp_wrapped)
    for n, p in _m.encoder.named_parameters():                      # only LoRA matrices may be trainable inside the encoder
        assert p.requires_grad == ('lora_A' in n or 'lora_B' in n), n
    if _name == 'E1_hier':
        # padding invariance: batched/padded sentence vectors == one-by-one vectors
        _m.eval()
        sample = [make_input(TRAIN_RECS[0], i, 1) for i in range(6)]
        with torch.no_grad():
            batched = _m.encode_sentences(sample)
            single = torch.cat([_m.encode_sentences([s]) for s in sample])
        assert len({len(s[0]) for s in sample}) > 1, 'test needs sentences of different lengths'
        err = (batched - single).abs().max().item()
        assert err < 2e-3, f'padding leaks into the encoder (max diff {err:.2e})'
        # LoRA starts as the identity: adapted encoder == plain Phase 1 encoder
        base = LegalEncoder(cfg).to(device); load_phase1_encoder_weights(base, phase1_state); base.eval()
        x = torch.tensor([sample[0][0]], device=device)
        with torch.no_grad():
            d0 = (base(x) - _m.encoder(x)).abs().max().item()
        assert d0 < 1e-4, f'LoRA is not an identity at init (diff {d0:.2e})'
        del base
        print(f'padding-invariance max diff {err:.1e} | LoRA-at-init max diff {d0:.1e}')
    _m.train()
    bw = pack_batches(make_train_windows(TRAIN_RECS, _hp['window'], _rng), _hp['batch_sentences'], _rng, True)[0]
    batch = collate_windows(TRAIN_RECS, bw, _m.ctx_k)
    with make_autocast():
        logits = model_forward(_m, batch)
    assert logits.shape == (len(batch['labels']), NUM_LABELS), logits.shape
    loss = F.cross_entropy(logits.float(), batch['labels'].to(device))
    loss.backward()
    g_enc, g_head = _grad_sums(_m)
    assert g_head > 0 and (g_enc > 0 if _hp['lora_r'] > 0 else g_enc == 0), (g_enc, g_head)
    print(f'{_name:16s} params {tot:,} (trainable {trn:,}) | LoRA layers {_m.n_lora_layers} | batch {len(batch["labels"])} sentences '
          f'| loss {loss.item():.3f} | grad enc {g_enc:.1f} head {g_head:.1f} | peak {torch.cuda.max_memory_allocated()/1024**3:.2f} GB')
    del _m, logits, loss, batch
    free_gpu()
print('Smoke tests passed.')

padding-invariance max diff 9.5e-06 | LoRA-at-init max diff 0.0e+00
E1_hier          params 146,808,622 (trainable 7,719,982) | LoRA layers 112 | batch 128 sentences | loss 2.967 | grad enc 490.8 head 7097.9 | peak 1.59 GB
E3_tokctx1       params 146,808,622 (trainable 7,719,982) | LoRA layers 112 | batch 128 sentences | loss 3.154 | grad enc 498.1 head 6959.7 | peak 2.82 GB
E5_frozen_probe  params 143,335,214 (trainable 4,246,574) | LoRA layers 0 | batch 98 sentences | loss 2.651 | grad enc 0.0 head 6296.7 | peak 2.82 GB
Smoke tests passed.


## 13. Run the experiments (finished runs with an identical config are reloaded from Drive)

In [19]:
RESULTS = globals().get('RESULTS', {})

def run_or_load(name, hp, seed=SEED):
    run_dir = EXPERIMENTS_ROOT / f'{name}_seed{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    sp, lp = run_dir / 'summary.json', run_dir / 'dev_logp_best.pkl'
    if sp.is_file() and lp.is_file():
        with open(sp) as f:
            s = json.load(f)
        if s.get('hp') == json.loads(json.dumps(hp)) and s.get('seed') == seed:
            with open(lp, 'rb') as f:
                s['best_logp'] = pickle.load(f)
            print(f'Loaded finished run from Drive: {name}')
            return s
    model, info = fit_model(hp, TRAIN_RECS, DEV_SETS, seed=seed, tag=name)
    torch.save({'trainable_state': info['best_state'], 'hp': hp, 'best_epoch': info['best_epoch']}, run_dir / 'best_trainable.pt')
    with open(lp, 'wb') as f:
        pickle.dump(info['best_logp'], f)
    s = {k: v for k, v in info.items() if k not in ('best_state', 'best_logp', 'last_logp')}
    s.update(hp=json.loads(json.dumps(hp)), seed=seed, name=name)
    with open(sp, 'w') as f:
        json.dump(s, f, indent=1)
    s['best_logp'] = info['best_logp']
    del model, info
    free_gpu()
    return s

for name, hp in EXPERIMENTS.items():
    if RUN_ONLY and name not in RUN_ONLY:
        continue
    RESULTS[name] = run_or_load(name, hp)

[E0_sentence_only] params total=143,899,438 trainable=4,810,798 (3.34%) | LoRA layers=112 | steps=1140 | class weights=off
[E0_sentence_only] epoch=1 | train_loss=2.0595 | train_seconds=64.3074 | cl_macro_f1=0.1415 | it_macro_f1=0.2229 | mean_macro_f1=0.1822 | cl_macro_f1_13=0.1197 | it_macro_f1_13=0.1886 | cl_acc=0.3345 | it_acc=0.4209
[E0_sentence_only] epoch=2 | train_loss=1.6960 | train_seconds=64.2443 | cl_macro_f1=0.2403 | it_macro_f1=0.3005 | mean_macro_f1=0.2704 | cl_macro_f1_13=0.1849 | it_macro_f1_13=0.2542 | cl_acc=0.3632 | it_acc=0.4196
[E0_sentence_only] epoch=3 | train_loss=1.5144 | train_seconds=64.3586 | cl_macro_f1=0.2718 | it_macro_f1=0.3998 | mean_macro_f1=0.3358 | cl_macro_f1_13=0.2091 | it_macro_f1_13=0.3383 | cl_acc=0.3775 | it_acc=0.4678
[E0_sentence_only] epoch=4 | train_loss=1.3400 | train_seconds=63.3606 | cl_macro_f1=0.2766 | it_macro_f1=0.4173 | mean_macro_f1=0.3469 | cl_macro_f1_13=0.2340 | it_macro_f1_13=0.3531 | cl_acc=0.4076 | it_acc=0.4786
[E0_sentence_

In [20]:
GQA_V2_REF = dict(CL=0.4541, IT=0.4847, mean=0.4694)      # best GQA-decoder dev result (D3) from the v2 notebook, same metric

rows = []
for name, s in RESULTS.items():
    m = s['best_metrics']
    rows.append(dict(experiment=name, best_epoch=s['best_epoch'],
                     CL_macroF1=m['CL']['macro_f1'], IT_macroF1=m['IT']['macro_f1'],
                     mean_macroF1=(m['CL']['macro_f1'] + m['IT']['macro_f1']) / 2,
                     mean_macroF1_13label_old_metric=(m['CL']['macro_f1_13'] + m['IT']['macro_f1_13']) / 2,
                     CL_macroP=m['CL']['macro_precision'], IT_macroP=m['IT']['macro_precision'],
                     CL_macroR=m['CL']['macro_recall'], IT_macroR=m['IT']['macro_recall'],
                     CL_acc=m['CL']['accuracy'], IT_acc=m['IT']['accuracy'],
                     trainable_params=s['trainable_params'], minutes=s['minutes'], peak_gb=s['peak_gb']))
results_df = pd.DataFrame(rows).sort_values('mean_macroF1', ascending=False).reset_index(drop=True)
results_df['delta_vs_GQA_v2_best'] = results_df['mean_macroF1'] - GQA_V2_REF['mean']
display(results_df.round(4))
results_df.to_csv(PROJECT_DIR / 'v3_encoder_experiment_comparison.csv', index=False)
BEST_NAME = results_df.iloc[0]['experiment']
print('Best configuration on dev:', BEST_NAME)
print(f'GQA-decoder v2 best (dev): CL {GQA_V2_REF["CL"]:.4f}  IT {GQA_V2_REF["IT"]:.4f}  mean {GQA_V2_REF["mean"]:.4f}')
print('Reminder: the dev sets are 5 documents per domain; differences below ~0.02-0.03 are within noise.')

,experiment,best_epoch,CL_macroF1,IT_macroF1,mean_macroF1,mean_macroF1_13label_old_metric,CL_macroP,IT_macroP,CL_macroR,IT_macroR,CL_acc,IT_acc,trainable_params,minutes,peak_gb,delta_vs_GQA_v2_best
0,E4_tokctx1_cw,7,0.4534,0.5349,0.4942,0.4181,0.4694,0.6089,0.4658,0.5101,0.6397,0.5925,7719982,10.4596,2.9202,0.0248
1,E6_win64_cw,8,0.4157,0.5350,0.4753,0.4022,0.4410,0.5930,0.4124,0.5141,0.5623,0.5804,7719982,9.9052,1.7398,0.0059
2,E3_tokctx1,7,0.4364,0.4962,0.4663,0.3946,0.4585,0.5857,0.4456,0.4667,0.6168,0.5831,7719982,10.3833,2.9177,-0.0031
3,E1_hier,4,0.4481,0.4808,0.4644,0.3758,0.4538,0.6374,0.4615,0.4645,0.5795,0.5590,7719982,9.1541,1.7306,-0.0050
4,E2_hier_cw,4,0.4186,0.5084,0.4635,0.3922,0.4096,0.5946,0.4340,0.5108,0.5702,0.5831,7719982,9.1887,1.7303,-0.0059
5,E5_frozen_probe,8,0.3926,0.4840,0.4383,0.3859,0.4153,0.5953,0.3844,0.4636,0.5953,0.5737,4246574,1.4574,0.7978,-0.0311
6,E0_sentence_only,5,0.3078,0.3914,0.3496,0.2958,0.3573,0.5103,0.3305,0.3777,0.4083,0.4598,4810798,8.7204,1.6886,-0.1198


Best configuration on dev: E4_tokctx1_cw
GQA-decoder v2 best (dev): CL 0.4541  IT 0.4847  mean 0.4694
Reminder: the dev sets are 5 documents per domain; differences below ~0.02-0.03 are within noise.


In [21]:
# logit adjustment (subtract tau * log prior); earlier runs selected tau = 0, which is a fine outcome
LOG_PRIOR = train_log_prior(TRAIN_RECS)
best_logp = RESULTS[BEST_NAME]['best_logp']
TAU, tau_df = tune_tau(best_logp, DEV_SETS, LOG_PRIOR)
display(tau_df.round(4))
print('Selected tau =', TAU)
adj = domain_metrics(best_logp, DEV_SETS, TAU, LOG_PRIOR)
raw = domain_metrics(best_logp, DEV_SETS, 0.0, LOG_PRIOR)
print(f'raw     : CL {raw["CL"]["macro_f1"]:.4f}  IT {raw["IT"]["macro_f1"]:.4f}  mean {mean_f1(raw):.4f}')
print(f'adjusted: CL {adj["CL"]["macro_f1"]:.4f}  IT {adj["IT"]["macro_f1"]:.4f}  mean {mean_f1(adj):.4f}')

# error analysis of the best configuration (dev)
for dom in ['CL', 'IT']:
    recs = DEV_SETS[dom]
    lp = np.concatenate(best_logp[dom]) - TAU * LOG_PRIOR
    y_true = np.concatenate([r['labels'] for r in recs]); y_pred = lp.argmax(1)
    print(f'\n===== {dom} dev: per-class scores =====')
    display(pd.DataFrame(rr_metrics(y_true, y_pred)['per_class']).round(3))
    cm = confusion_matrix(y_true, y_pred, labels=ALL_IDS)
    pairs = [(cm[i, j], id_to_label[i], id_to_label[j]) for i in ALL_IDS for j in ALL_IDS if i != j and cm[i, j] > 0]
    print(f'{dom}: most frequent confusions (gold -> predicted):')
    for c, a, b in sorted(pairs, reverse=True)[:8]:
        print(f'   {a:24s} -> {b:24s} {c}')

,tau,CL,IT,mean
0,0.00,0.4534,0.5349,0.4942
1,0.25,0.4679,0.5377,0.5028
2,0.50,0.4286,0.4874,0.4580
3,0.75,0.3024,0.3453,0.3239
4,1.00,0.1032,0.2008,0.1520
5,1.25,0.0611,0.1295,0.0953


Selected tau = 0.25
raw     : CL 0.4534  IT 0.5349  mean 0.4942
adjusted: CL 0.4679  IT 0.5377  mean 0.5028

===== CL dev: per-class scores =====


,label,precision,recall,f1,support
0,Fact,0.581,0.651,0.614,249
1,Issue,0.143,0.333,0.200,3
2,ArgumentPetitioner,0.753,0.523,0.617,344
3,ArgumentRespondent,0.673,0.716,0.694,236
4,Statute,0.694,0.862,0.769,58
5,Dissent,0.000,0.000,0.000,0
6,PrecedentReliedUpon,0.569,0.493,0.528,142
7,PrecedentNotReliedUpon,0.000,0.000,0.000,0
8,PrecedentOverruled,0.000,0.000,0.000,0
9,RulingByLowerCourt,0.607,0.304,0.405,56


CL: most frequent confusions (gold -> predicted):
   ArgumentPetitioner       -> ArgumentRespondent       80
   Fact                     -> RatioOfTheDecision       44
   PrecedentReliedUpon      -> Fact                     39
   RulingByLowerCourt       -> Fact                     29
   ArgumentRespondent       -> ArgumentPetitioner       29
   PrecedentReliedUpon      -> RatioOfTheDecision       27
   ArgumentPetitioner       -> Fact                     26
   RatioOfTheDecision       -> PrecedentReliedUpon      22

===== IT dev: per-class scores =====


,label,precision,recall,f1,support
0,Fact,0.801,0.703,0.749,195
1,Issue,0.667,0.667,0.667,12
2,ArgumentPetitioner,0.542,0.571,0.557,56
3,ArgumentRespondent,0.571,0.222,0.320,36
4,Statute,0.852,0.742,0.793,62
5,Dissent,0.000,0.000,0.000,0
6,PrecedentReliedUpon,0.507,0.616,0.556,112
7,PrecedentNotReliedUpon,0.298,0.340,0.318,50
8,PrecedentOverruled,0.000,0.000,0.000,0
9,RulingByLowerCourt,0.468,0.478,0.473,46


IT: most frequent confusions (gold -> predicted):
   Fact                     -> RatioOfTheDecision       27
   RatioOfTheDecision       -> PrecedentReliedUpon      23
   PrecedentReliedUpon      -> RatioOfTheDecision       21
   RulingByPresentCourt     -> RatioOfTheDecision       18
   PrecedentNotReliedUpon   -> PrecedentReliedUpon      18
   Fact                     -> RulingByLowerCourt       17
   None                     -> PrecedentReliedUpon      14
   ArgumentRespondent       -> ArgumentPetitioner       13


## 14. Final refit on train + dev, multi-seed ensemble, test predictions
The best configuration (by dev macro-F1) is retrained from the Phase 1 checkpoint on **train + dev** for the same fixed number of epochs (the cosine schedule ends near zero LR, so the last epoch is used), with several seeds. Test log-probabilities are averaged over seeds, τ-adjusted, then arg-maxed. To submit a *different* run than the top row (e.g. the dev ranking is a near-tie), set `BEST_NAME` explicitly in the next cell.

In [22]:
# BEST_NAME = 'E3_tokctx1'          # <- uncomment to override the dev-selected configuration
FINAL_SEEDS = [42, 43, 44]          # more seeds = more stable; each costs about one experiment's training time
FINAL_HP = EXPERIMENTS[BEST_NAME]
ALL_LABELED = TRAIN_RECS + REC['cl_dev'] + REC['it_dev']
TEST_SETS = {'CL': REC['cl_test'], 'IT': REC['it_test']}
FINAL_LOG_PRIOR = train_log_prior(ALL_LABELED)
fp = hashlib.md5(json.dumps(FINAL_HP, sort_keys=True).encode()).hexdigest()[:8]
print('Final configuration:', BEST_NAME, '| fingerprint', fp)

seed_test_logp, final_train_stats = {}, []
for sd in FINAL_SEEDS:
    cache = FINAL_ARTIFACT_DIR / f'test_logp_seed{sd}_{fp}.pkl'
    if cache.is_file():
        with open(cache, 'rb') as f:
            payload = pickle.load(f)
        seed_test_logp[sd] = payload['logp']; final_train_stats.append(payload['stats'])
        print(f'Loaded cached final seed {sd}')
        continue
    model, info = fit_model(FINAL_HP, ALL_LABELED, None, seed=sd, tag=f'final-seed{sd}')
    t0 = time.perf_counter()
    logp = {d: predict_logp(model, recs, FINAL_HP['window'], FINAL_HP['eval_stride']) for d, recs in TEST_SETS.items()}
    n_sent = sum(r['n'] for recs in TEST_SETS.values() for r in recs)
    stats = dict(seed=sd, minutes=info['minutes'], peak_gb=info['peak_gb'], trainable_params=info['trainable_params'],
                 total_params=info['total_params'], test_inference_seconds=time.perf_counter() - t0,
                 test_sentences=n_sent, final_train_loss=info['history'][-1]['train_loss'])
    torch.save({'trainable_state': trainable_state(model), 'hp': FINAL_HP, 'seed': sd, 'name': BEST_NAME},
               FINAL_ARTIFACT_DIR / f'final_trainable_seed{sd}_{fp}.pt')
    with open(cache, 'wb') as f:
        pickle.dump(dict(logp=logp, stats=stats), f)
    seed_test_logp[sd] = logp; final_train_stats.append(stats)
    del model, info
    free_gpu()

def ensemble_logp(per_seed, dom):
    """Average probabilities over seeds, return the log of the mean probability per document."""
    seeds = list(per_seed)
    out = []
    for di in range(len(per_seed[seeds[0]][dom])):
        probs = np.mean([np.exp(per_seed[s][dom][di]) for s in seeds], axis=0)
        out.append(np.log(probs + 1e-12))
    return out

ens = {d: ensemble_logp(seed_test_logp, d) for d in TEST_SETS}
print('Ensemble built from seeds:', list(seed_test_logp))

Final configuration: E4_tokctx1_cw | fingerprint dbb7727c
[final-seed42] params total=146,808,622 trainable=7,719,982 (5.26%) | LoRA layers=112 | steps=1289 | class weights=on
[final-seed42] epoch=1 | train_loss=2.2436 | train_seconds=83.0665
[final-seed42] epoch=2 | train_loss=1.7297 | train_seconds=82.9293
[final-seed42] epoch=3 | train_loss=1.5006 | train_seconds=83.1069
[final-seed42] epoch=4 | train_loss=1.2775 | train_seconds=83.3357
[final-seed42] epoch=5 | train_loss=1.0398 | train_seconds=83.3571
[final-seed42] epoch=6 | train_loss=0.8321 | train_seconds=83.2030
[final-seed42] epoch=7 | train_loss=0.6900 | train_seconds=83.1358
[final-seed42] epoch=8 | train_loss=0.6244 | train_seconds=83.2010
[final-seed43] params total=146,808,622 trainable=7,719,982 (5.26%) | LoRA layers=112 | steps=1290 | class weights=on
[final-seed43] epoch=1 | train_loss=2.1991 | train_seconds=83.6425
[final-seed43] epoch=2 | train_loss=1.7593 | train_seconds=82.8967
[final-seed43] epoch=3 | train_loss=

In [25]:
# Format (spec 9.3.5): {"document_id": ["Fact", "Issue", ...]}
def predictions_for(dom, docs):
    preds = {}
    for doc, lp in zip(docs, ens[dom]):
        lab = (lp - TAU * FINAL_LOG_PRIOR).argmax(1)
        preds[str(doc['id'])] = [id_to_label[int(i)] for i in lab]
        assert len(preds[str(doc['id'])]) == len(doc['text'])
    return preds

cl_predictions = predictions_for('CL', cl_test_docs)
it_predictions = predictions_for('IT', it_test_docs)

overlap = set(cl_predictions) & set(it_predictions)
if overlap:
    print(f'WARNING: {len(overlap)} document IDs appear in both CL and IT test sets; submit the two files separately. Combined file NOT written.')
    flat = None
else:
    flat = {**cl_predictions, **it_predictions}

CL_PREDICTIONS_PATH = FINAL_ARTIFACT_DIR / 'CL_test_predictions.json'
IT_PREDICTIONS_PATH = FINAL_ARTIFACT_DIR / 'IT_test_predictions.json'
COMBINED_SUBMISSION_PATH = FINAL_ARTIFACT_DIR / 'submission.json'
to_write = [(CL_PREDICTIONS_PATH, cl_predictions), (IT_PREDICTIONS_PATH, it_predictions)]
if flat is not None:
    to_write.append((COMBINED_SUBMISSION_PATH, flat))
for path, obj in to_write:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def validate(path, docs_list, name):
    with open(path, encoding='utf-8') as f:
        p = json.load(f)
    docs = [d for dl in docs_list for d in dl]
    assert list(p.keys()) == [str(d['id']) for d in docs], f'{name}: id/order mismatch'
    valid, total = set(id_to_label), 0
    for d in docs:
        labs = p[str(d['id'])]
        assert isinstance(labs, list) and len(labs) == len(d['text']), f'{name}: {d["id"]} length mismatch'
        assert all(l in valid for l in labs), f'{name}: {d["id"]} has an invalid label'
        total += len(labs)
    return dict(documents=len(docs), sentences=total)

cl_val = validate(CL_PREDICTIONS_PATH, [cl_test_docs], 'CL')
it_val = validate(IT_PREDICTIONS_PATH, [it_test_docs], 'IT')
print('CL:', cl_val, '| IT:', it_val)
if flat is not None:
    print('Combined flat file:', validate(COMBINED_SUBMISSION_PATH, [cl_test_docs, it_test_docs], 'COMBINED'))

for name, p in [('CL', cl_predictions), ('IT', it_predictions)]:
    c = Counter(l for labs in p.values() for l in labs)
    print(f'\n{name} predicted label distribution:')
    for l in id_to_label:
        print(f'   {l:24s} {c[l]:6d}')

SUBMISSION_ZIP_PATH = FINAL_ARTIFACT_DIR / 'phase3_rr_encoder_lora_submission.zip'
with zipfile.ZipFile(SUBMISSION_ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(CL_PREDICTIONS_PATH, arcname='CL_test_predictions.json')
    zf.write(IT_PREDICTIONS_PATH, arcname='IT_test_predictions.json')
    if flat is not None:
        zf.write(COMBINED_SUBMISSION_PATH, arcname='submission.json')
print('\nSubmission zip:', SUBMISSION_ZIP_PATH)

CL: {'documents': 5, 'sentences': 1339} | IT: {'documents': 5, 'sentences': 858}
Combined flat file: {'documents': 10, 'sentences': 2197}

CL predicted label distribution:
   Fact                        316
   Issue                         4
   ArgumentPetitioner          148
   ArgumentRespondent           62
   Statute                      71
   Dissent                      23
   PrecedentReliedUpon         146
   PrecedentNotReliedUpon       20
   PrecedentOverruled            0
   RulingByLowerCourt           46
   RatioOfTheDecision          475
   RulingByPresentCourt         19
   None                          9

IT predicted label distribution:
   Fact                        132
   Issue                        14
   ArgumentPetitioner           52
   ArgumentRespondent           56
   Statute                     122
   Dissent                       0
   PrecedentReliedUpon         141
   PrecedentNotReliedUpon        8
   PrecedentOverruled            0
   RulingByLowerCourt   

In [24]:
sel = RESULTS[BEST_NAME]['best_metrics']
report = {
    'project': 'Phase 3 RR - Phase 1 encoder-only (bidirectional GQA, MLM) + LoRA, hierarchical document model (v3)',
    'phase1_checkpoint': str(PHASE1_CHECKPOINT_PATH),
    'selected_configuration': BEST_NAME,
    'hyperparameters': FINAL_HP,
    'tau_logit_adjustment': TAU,
    'final_seeds': FINAL_SEEDS,
    'dev_metrics_selected_config_single_seed': {d: {k: v for k, v in sel[d].items() if k != 'per_class'} for d in sel},
    'dev_metrics_after_logit_adjustment': {d: {k: v for k, v in adj[d].items() if k != 'per_class'} for d in adj},
    'gqa_decoder_v2_reference_dev': GQA_V2_REF,
    'experiment_comparison': json.loads(results_df.to_json(orient='records')),
    'final_training_stats': final_train_stats,
    'test_validation': {'CL': cl_val, 'IT': it_val},
    'limitations': [
        'Dev sets are tiny (5 documents per domain) and miss several rare labels; dev F1 is noisy (+-0.02 or more).',
        'The final models are refit on train + dev, so no dev score exists for the exact submitted ensemble.',
        'PrecedentOverruled has 3 training sentences (IT only) and Dissent occurs only in CL; these cannot be learned reliably.',
    ],
}
with open(FINAL_ARTIFACT_DIR / 'v3_encoder_final_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

md_lines = ['# Phase 3 RR - Phase 1 encoder + LoRA (v3, hierarchical)', '',
            f'- Selected configuration: `{BEST_NAME}`',
            f'- LoRA: r={FINAL_HP["lora_r"]}, alpha={FINAL_HP["lora_alpha"]}, rsLoRA={FINAL_HP["rslora"]}, targets={FINAL_HP["target_modules"]}',
            f'- Document window: {FINAL_HP["window"]} sentences (BiLSTM); in-encoder neighbour sentences ctx_k={FINAL_HP["ctx_k"]}; position features: {FINAL_HP["use_position"]}',
            f'- Class-weight power: {FINAL_HP["cw_power"]}; logit-adjustment tau: {TAU}; ensemble seeds: {FINAL_SEEDS}', '',
            '## Dev macro-F1 (sklearn default macro; single seed, train-only)',
            f'- CL: {sel["CL"]["macro_f1"]:.4f}   IT: {sel["IT"]["macro_f1"]:.4f}   mean: {mean_f1(sel):.4f}',
            f'- GQA decoder v2 reference: CL {GQA_V2_REF["CL"]:.4f}   IT {GQA_V2_REF["IT"]:.4f}   mean {GQA_V2_REF["mean"]:.4f}', '',
            '## Ablation table', '```', results_df.round(4).to_string(index=False), '```', '',
            '## Test predictions', f'- CL: {cl_val}', f'- IT: {it_val}', '- Test labels are private; no test F1 is reported.']
(FINAL_ARTIFACT_DIR / 'v3_encoder_final_report.md').write_text('\n'.join(md_lines) + '\n', encoding='utf-8')
print('Saved report to', FINAL_ARTIFACT_DIR)
for p in sorted(FINAL_ARTIFACT_DIR.iterdir()):
    if p.is_file():
        print(f'{p.name:60s} {p.stat().st_size / 1024:10.1f} KiB')

Saved report to /content/drive/MyDrive/phase3_rr_encoder_lora/final_submission
CL_test_predictions.json                                           29.2 KiB
IT_test_predictions.json                                           18.6 KiB
final_trainable_seed42_dbb7727c.pt                              30241.9 KiB
final_trainable_seed43_dbb7727c.pt                              30241.9 KiB
final_trainable_seed44_dbb7727c.pt                              30241.9 KiB
phase3_rr_encoder_lora_submission.zip                               3.3 KiB
submission.json                                                    47.8 KiB
test_logp_seed42_dbb7727c.pkl                                     223.8 KiB
test_logp_seed43_dbb7727c.pkl                                     223.8 KiB
test_logp_seed44_dbb7727c.pkl                                     223.8 KiB
v3_encoder_final_report.json                                        7.5 KiB
v3_encoder_final_report.md                                          2.5 KiB
